# FontDiffusion — Sinh ảnh chữ Nôm viết tay cổ

Script sinh ảnh cho **tất cả** ứng viên tier 3 từ 3 bộ dữ liệu CacThanhTruyen 2, 4, 11.

**Flow:**
1. Cài đặt + clone repo + download model
2. Upload `all_candidates.txt` + `style_reference.png`
3. Sinh ảnh batch trên GPU
4. Download `fd_cache.zip` về máy local

**Yêu cầu:** Runtime → GPU (T4 hoặc A100)

## 1. Cài đặt thư viện

In [ ]:
!pip install -q torch torchvision diffusers accelerate safetensors
!pip install -q einops info-nce-pytorch kornia pygame Pillow huggingface-hub

## 2. Clone FontDiffusion repo

In [ ]:
!git clone https://github.com/dzungphieuluuky/FontDiffusion.git
%cd FontDiffusion

## 3. Download model checkpoint từ HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "dzungpham/font-architect"
CKPT_DIR = "ckpt"

files_to_download = [
    # FST Phase 1
    "FST/checkpoint_step_1500/content_encoder.safetensors",
    "FST/checkpoint_step_1500/fst_module.safetensors",
    "FST/checkpoint_step_1500/fst_projection.safetensors",
    "FST/checkpoint_step_1500/mss_encoder.safetensors",
    "FST/checkpoint_step_1500/original_style_projection.safetensors",
    "FST/checkpoint_step_1500/style_encoder.safetensors",
    "FST/checkpoint_step_1500/unet.safetensors",
    # DRO Phase 2 (latest)
    "DRO-20260227-19P2/checkpoint_step_6000/content_encoder.safetensors",
    "DRO-20260227-19P2/checkpoint_step_6000/fst_module.safetensors",
    "DRO-20260227-19P2/checkpoint_step_6000/fst_projection.safetensors",
    "DRO-20260227-19P2/checkpoint_step_6000/mss_encoder.safetensors",
    "DRO-20260227-19P2/checkpoint_step_6000/original_style_projection.safetensors",
    "DRO-20260227-19P2/checkpoint_step_6000/scr.safetensors",
    "DRO-20260227-19P2/checkpoint_step_6000/style_encoder.safetensors",
    "DRO-20260227-19P2/checkpoint_step_6000/unet.safetensors",
]

for i, f in enumerate(files_to_download):
    print(f"[{i+1}/{len(files_to_download)}] {f}")
    hf_hub_download(repo_id=REPO, filename=f, local_dir=CKPT_DIR)

print("Done!")

## 4. Upload files

Upload 2 file:
- `all_candidates.txt` — danh sách 6409 ký tự cần sinh
- `style_reference.png` — 1 ảnh crop chữ Nôm viết tay làm mẫu phong cách

In [ ]:
from google.colab import files
uploaded = files.upload()
print(f"Uploaded: {list(uploaded.keys())}")

## 5. Kiểm tra trước khi chạy

In [ ]:
import torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Check files
assert Path("all_candidates.txt").exists(), "Missing all_candidates.txt!"
assert Path("style_reference.png").exists(), "Missing style_reference.png!"

with open("all_candidates.txt", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]
print(f"Candidates: {len(lines)}")
print(f"Sample: {[l.split(chr(9))[0] for l in lines[:10]]}")

# Show style image
img = Image.open("style_reference.png")
plt.figure(figsize=(2, 2))
plt.imshow(img, cmap="gray")
plt.title("Style Reference")
plt.axis("off")
plt.show()

## 6. Sinh ảnh (MAIN)

In [ ]:
import os, sys, time
from pathlib import Path
import torch
from PIL import Image

# ── Config ──
CKPT_DIR = "ckpt/DRO-20260227-19P2/checkpoint_step_6000"
PHASE1_CKPT_DIR = "ckpt/FST/checkpoint_step_1500"
FONT_PATH = "fonts/NomNaTong-Regular.ttf"
STYLE_IMAGE = "style_reference.png"
CANDIDATES_FILE = "all_candidates.txt"
OUTPUT_DIR = Path("fd_cache")
BATCH_SIZE = 16  # T4: 16, A100: 32

# ── Read candidates ──
candidates = []
with open(CANDIDATES_FILE, "r", encoding="utf-8") as f:
    for line in f:
        char = line.strip().split("\t")[0].strip()
        if char:
            candidates.append(char)

# ── Check cache ──
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
to_generate = [c for c in candidates if not (OUTPUT_DIR / f"U+{ord(c):04X}.png").exists()]
print(f"Total: {len(candidates)}, Cached: {len(candidates)-len(to_generate)}, To generate: {len(to_generate)}")

if not to_generate:
    print("All cached! Skip to download.")
else:
    # ── Load pipeline (FP32 to avoid dtype mismatch) ──
    sys.path.insert(0, ".")
    from src.configs.fontdiffuser import get_parser
    from inference.sample_optimized import (
        load_fontdiffuser_pipeline, FontManager,
        get_style_transform, get_content_transform,
    )
    from src.tools.utils import ttf2im

    parser = get_parser()
    args = parser.parse_args([])
    args.ckpt_dir = CKPT_DIR
    args.phase_1_ckpt_dir = PHASE1_CKPT_DIR
    args.fst_ckpt_path = CKPT_DIR
    args.ttf_path = FONT_PATH
    args.device = "cuda" if torch.cuda.is_available() else "cpu"
    args.use_fst = True
    args.batch_size = BATCH_SIZE
    args.fp16 = False  # Load model in FP32 to avoid dtype mismatch
    args.style_image_size = (args.style_image_size, args.style_image_size)
    args.content_image_size = (args.content_image_size, args.content_image_size)

    print(f"Device: {args.device}")
    print("Loading pipeline (FP32)...")
    pipe = load_fontdiffuser_pipeline(args=args, use_fst=True)

    # ── Prepare data ──
    font_manager = FontManager(FONT_PATH)
    font_name = font_manager.get_font_names()[0]
    font = font_manager.get_font(font_name)
    style_transform = get_style_transform(args.style_image_size)
    content_transform = get_content_transform(args.content_image_size)

    style_image = Image.open(STYLE_IMAGE).convert("RGB")
    style_tensor = style_transform(style_image)

    available = font_manager.get_available_chars_for_font(font_name, to_generate)
    print(f"Available in font: {len(available)}/{len(to_generate)}")

    content_tensors, valid_chars = [], []
    for char in available:
        try:
            img = ttf2im(font=font, char=char)
            if img is not None:
                content_tensors.append(content_transform(img))
                valid_chars.append(char)
        except:
            continue

    print(f"Ready: {len(valid_chars)} characters")

    # ── Generate (FP32 throughout) ──
    content_batch = torch.stack(content_tensors)
    style_batch = style_tensor[None, :].repeat(len(content_tensors), 1, 1, 1)
    content_batch = content_batch.to(args.device, dtype=torch.float32)
    style_batch = style_batch.to(args.device, dtype=torch.float32)

    print(f"\n{'='*60}")
    print(f"Generating {len(valid_chars)} images (batch={BATCH_SIZE})...")
    print(f"{'='*60}")

    start = time.time()
    all_images = []

    with torch.inference_mode():
        for i in range(0, len(content_batch), BATCH_SIZE):
            bc = content_batch[i:i + BATCH_SIZE]
            bs = style_batch[i:i + BATCH_SIZE]
            images = pipe.generate(
                content_images=bc, style_images=bs,
                batch_size=len(bc), order=args.order,
                num_inference_step=args.num_inference_steps,
                content_encoder_downsample_size=args.content_encoder_downsample_size,
                t_start=None, t_end=None,
                dm_size=args.content_image_size,
                algorithm_type=args.algorithm_type,
                skip_type=args.skip_type,
                method=args.method,
                correcting_x0_fn=None,
            )
            all_images.extend(images)

            done = min(i + BATCH_SIZE, len(valid_chars))
            elapsed = time.time() - start
            eta = (elapsed / done) * (len(valid_chars) - done)
            print(f"  [{done:>5}/{len(valid_chars)}] {elapsed:>6.0f}s elapsed, ~{eta:>5.0f}s remaining")

    total = time.time() - start
    print(f"\nDone! {len(all_images)} images in {total:.0f}s ({total/max(1,len(all_images)):.2f}s/img)")

    # ── Save ──
    for char, img in zip(valid_chars, all_images):
        img.save(str(OUTPUT_DIR / f"U+{ord(char):04X}.png"))

    print(f"Saved: {len(list(OUTPUT_DIR.glob('U+*.png')))} files")

## 7. Xem thử kết quả

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

samples = sorted(Path("fd_cache").glob("U+*.png"))[:12]
if samples:
    fig, axes = plt.subplots(2, 6, figsize=(15, 5))
    for ax, p in zip(axes.flat, samples):
        img = Image.open(p)
        code = p.stem  # U+XXXX
        char = chr(int(code.replace("U+", ""), 16))
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{char} ({code})", fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No images found")

## 8. Download kết quả

Download `fd_cache.zip`, rồi trên máy local:
```bash
unzip fd_cache.zip -d prepared/CacThanhTruyen2/fd_cache/
cp -r prepared/CacThanhTruyen2/fd_cache prepared/CacThanhTruyen4/fd_cache
cp -r prepared/CacThanhTruyen2/fd_cache prepared/CacThanhTruyen11/fd_cache
./run_pipeline.sh --step 3
./run_pipeline.sh --step 4
```

In [ ]:
import shutil
shutil.make_archive("fd_cache", "zip", ".", "fd_cache")
print(f"fd_cache.zip created")

from google.colab import files
files.download("fd_cache.zip")